# 02 — Court Homography

**Camera context:** Broadcast shows the FAR end of the court (basket near CITGO sign).
 Need to annotate the paint box.

**4 keypoints (all in far paint box):**
- paint_BL: left lane line meets far baseline 
- paint_BR: right lane line meets far baseline 
- paint_FL: left corner of far free throw line 
- paint_FR: right corner of far free throw line 

In [2]:
import cv2
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path('/Users/neeldavuluri/uvabballcv')
FRAMES_DIR   = PROJECT_ROOT / 'frames'
HOMO_DIR     = PROJECT_ROOT / 'homography'
HOMO_DIR.mkdir(exist_ok=True)

# ═══════════════════════════════════════════════════
# CHANGE THIS PART FOR EACH CLIP AFTER ANNOTATION
ANNOTATE_CLIP = '10_uva_oreb_letup_block'

clicked_points = [
    [520,  504],
    [687,  387],
    [1090, 582],
    [1203, 448],
]

WORLD_COORDS = {
    'paint_BL': [17, 0],
    'paint_BR': [33, 0],
    'paint_FL': [17, 19],
    'paint_FR': [33, 19],
}
# ═══════════════════════════════════════════════════

POINT_ORDER = ['paint_BL', 'paint_BR', 'paint_FL', 'paint_FR']

# Save ref frame
frames = sorted((FRAMES_DIR / ANNOTATE_CLIP).glob('*.jpg'))
ref    = str(frames[len(frames)//2])
plt.imsave(str(PROJECT_ROOT / 'ref_frame.png'), plt.imread(ref))
print(f'ref_frame.png saved for {ANNOTATE_CLIP}')

# Skip compute if points are zeros
if clicked_points[0] == [0, 0]:
    print('Paste real coordinates then re-run.')
else:
    src = np.array(clicked_points, dtype=np.float32)
    dst = np.array([WORLD_COORDS[p] for p in POINT_ORDER], dtype=np.float32)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)

    if H is None:
        print('ERROR: homography failed.')
    else:
        print(f'Inliers: {int(mask.sum())}/4')
        for i, label in enumerate(POINT_ORDER):
            pt    = np.array([[[clicked_points[i][0], clicked_points[i][1]]]], dtype=np.float32)
            world = cv2.perspectiveTransform(pt, H)[0][0]
            print(f'  {label}: ({world[0]:.1f}, {world[1]:.1f})  expected: {WORLD_COORDS[label]}')

        out = {
            'clip'           : ANNOTATE_CLIP,
            'reference_frame': ref,
            'pixel_points'   : clicked_points,
            'world_points'   : [WORLD_COORDS[p] for p in POINT_ORDER],
            'point_labels'   : POINT_ORDER,
            'H'              : H.tolist(),
            'inliers'        : int(mask.sum())
        }
        with open(HOMO_DIR / f'{ANNOTATE_CLIP}_homography.json', 'w') as f:
            json.dump(out, f, indent=2)
        print(f'Saved -> {ANNOTATE_CLIP}_homography.json')

ref_frame.png saved for 10_uva_oreb_letup_block
Inliers: 4/4
  paint_BL: (17.0, 0.0)  expected: [17, 0]
  paint_BR: (33.0, 0.0)  expected: [33, 0]
  paint_FL: (17.0, 19.0)  expected: [17, 19]
  paint_FR: (33.0, 19.0)  expected: [33, 19]
Saved -> 10_uva_oreb_letup_block_homography.json


In [3]:
# Mark 11_uva_oreb_letup2 as unusable for homography
import json
with open(PROJECT_ROOT / 'clip_audit.json') as f:
    audit = json.load(f)
audit['11_uva_oreb_letup2']['usable'] = False
audit['11_uva_oreb_letup2']['notes'] = 'wide transition shot, paint not visible'
with open(PROJECT_ROOT / 'clip_audit.json', 'w') as f:
    json.dump(audit, f, indent=2)
print('11_uva_oreb_letup2 marked as unusable')

11_uva_oreb_letup2 marked as unusable


In [4]:
with open(PROJECT_ROOT / 'clip_audit.json') as f:
    audit = json.load(f)

usable = [name for name, v in audit.items() if v['usable'] is True]
done   = [name for name in usable if (HOMO_DIR / f'{name}_homography.json').exists()]
todo   = [name for name in usable if name not in done]

print(f'Done ({len(done)}):')
for c in done: print(f'  ✓ {c}')
print(f'\nTodo ({len(todo)}):')
for c in todo: print(f'  - {c}')

Done (8):
  ✓ 01_uva_steal
  ✓ 04_miami_midrange
  ✓ 05_uva_3pt
  ✓ 06_uva_midrange
  ✓ 08_uva_oreb_letup
  ✓ 09_uva_def_rebound2
  ✓ 10_uva_oreb_letup_block
  ✓ 12_uva_oreb_letup3

Todo (5):
  - 02_uva_def_rebound
  - 03_uva_pnr_dunk
  - 13_uva_layup
  - 14_miami_layup
  - 15_miami_layup2
